
# [SQL 실습 #03] MySQL 데이터 입력하기 — INSERT INTO 기초

> **학생용 실습 노트북 - TODO 완성형**

지난 시간에는 `school` 데이터베이스와 `students` 테이블을 만들었습니다.  
이번 시간에는 그 테이블 안에 **실제 데이터를 입력하는 방법**을 배웁니다.

---

## 오늘의 학습 목표

실습을 마치면 다음을 할 수 있어야 합니다.

1. `INSERT INTO` 기본 문법을 설명할 수 있다.
2. `students` 테이블에 한 줄의 데이터를 입력할 수 있다.
3. 여러 줄의 데이터를 한 번에 입력할 수 있다.
4. 입력한 데이터를 `SELECT`로 확인할 수 있다.
5. `AUTO_INCREMENT`의 동작을 확인할 수 있다.
6. 컬럼 순서와 값의 순서가 왜 중요한지 설명할 수 있다.
7. `NOT NULL` 오류가 발생하는 이유를 설명할 수 있다.
8. 일부 컬럼만 입력했을 때 `NULL`이 들어가는 이유를 이해할 수 있다.
9. `COUNT(*)`로 행 개수를 확인할 수 있다.
10. `TRUNCATE TABLE`의 역할을 설명할 수 있다.

---

## 오늘의 핵심

> **INSERT INTO는 테이블에 새로운 데이터를 추가하는 SQL 명령어입니다.**

쉽게 말하면,

> 테이블이라는 서류철을 만들어 두었다면,  
> `INSERT INTO`는 그 안에 실제 기록 한 장을 끼워 넣는 작업입니다.



# 1. 실습 환경 준비

코랩 런타임이 새로 시작되면 MySQL 서버와 지난 시간 데이터가 사라질 수 있습니다.

따라서 이번 노트북에서는 실습 시작 전에 다음 환경을 자동으로 준비합니다.

- MySQL 서버 설치
- MySQL 서버 실행
- `school` 데이터베이스 생성
- `students` 테이블 생성

> 아래 셀은 **수정하지 말고 그대로 실행**하세요.


In [ ]:
!sudo apt-get -qq update
!sudo DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!sudo service mysql start

print("✅ MySQL 서버 준비 완료")



# 2. 실습용 데이터베이스와 테이블 준비

이번 실습에서 사용할 구조는 다음과 같습니다.

| 컬럼 | 자료형 | 특징 |
|---|---|---|
| `id` | INT | AUTO_INCREMENT, PRIMARY KEY |
| `name` | VARCHAR(50) | NOT NULL |
| `grade` | INT | |
| `class_name` | VARCHAR(50) | |
| `created_at` | TIMESTAMP | DEFAULT CURRENT_TIMESTAMP |

### 꼭 기억하기

- `id`는 직접 입력하지 않아도 자동 증가합니다.
- `name`은 `NOT NULL`이므로 반드시 입력해야 합니다.
- `created_at`은 현재 시간이 자동 저장됩니다.

아래 셀은 실습 환경을 초기화합니다.


In [ ]:
%%bash
sudo mysql <<'SQL'
CREATE DATABASE IF NOT EXISTS school
DEFAULT CHARACTER SET utf8mb4
DEFAULT COLLATE utf8mb4_unicode_ci;

USE school;

DROP TABLE IF EXISTS students;

CREATE TABLE students (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    grade INT,
    class_name VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
SQL

echo "✅ school 데이터베이스와 students 테이블 준비 완료"



# 3. SQL 실행 도우미 함수

아래 함수는 여러 줄 SQL을 쉽게 실행하기 위한 도구입니다.

`TODO`가 남아 있으면 실행하지 않고 알려 줍니다.

> 이 셀은 수정하지 마세요.


In [ ]:
import subprocess
import textwrap

def run_sql(sql):
    sql = textwrap.dedent(sql).strip()

    # -- 로 시작하는 SQL 한 줄 주석을 제외하고 실제 SQL이 있는지 확인
    executable_lines = [
        line for line in sql.splitlines()
        if line.strip() and not line.lstrip().startswith("--")
    ]
    executable_sql = "\n".join(executable_lines).strip()

    if not executable_sql:
        print("⚠️ 아직 SQL이 작성되지 않았습니다. 설명을 읽고 직접 작성하세요.")
        return

    if "TODO" in executable_sql:
        print("⚠️ SQL 안에 TODO가 남아 있습니다. TODO를 모두 완성한 뒤 실행하세요.")
        return

    result = subprocess.run(
        ["sudo", "mysql"],
        input=sql,
        text=True,
        capture_output=True
    )

    if result.stdout:
        print(result.stdout)

    if result.returncode != 0:
        print("❌ MySQL 오류가 발생했습니다.")
        print(result.stderr.strip())
        print("\n💡 오류 메시지에서 데이터베이스명, 테이블명, 컬럼명, 따옴표를 확인하세요.")
        return

    if result.stderr.strip():
        print("ℹ️ MySQL 메시지:")
        print(result.stderr.strip())

    print("✅ SQL 실행 완료")



# 4. 사용할 데이터베이스와 테이블 확인

데이터를 넣기 전에 작업 대상이 맞는지 먼저 확인하는 습관이 좋습니다.

확인할 것:

1. `school` 데이터베이스 선택
2. 테이블 목록 확인
3. `students` 테이블 구조 확인

## TODO 1

아래 SQL을 완성하세요.


In [ ]:
sql = """
USE TODO;
SHOW TODO;
DESC TODO;
"""

run_sql(sql)



# 5. INSERT INTO 기본 문법

데이터 입력의 기본 형식은 다음과 같습니다.

```sql
INSERT INTO 테이블명 (컬럼1, 컬럼2, 컬럼3)
VALUES (값1, 값2, 값3);
```

핵심은 세 가지입니다.

1. **어느 테이블에 넣을지**
2. **어떤 컬럼에 넣을지**
3. **그 컬럼에 들어갈 값을 순서대로 적기**

예를 들어:

```sql
INSERT INTO students (name, grade, class_name)
VALUES ('김부장', 3, '컴퓨터과');
```

### 문자와 숫자의 차이

- 문자: 작은따옴표 `' '`로 감쌉니다.
- 숫자: 따옴표 없이 적습니다.

```text
'김부장'  → 문자
3         → 숫자
'컴퓨터과' → 문자
```



# 6. 데이터 한 줄 입력하기

첫 번째 학생 정보를 입력합니다.

- 이름: 김부장
- 학년: 3
- 반/학과: 컴퓨터과

## TODO 2

아래 TODO를 완성하세요.


In [ ]:
sql = """
USE school;

INSERT INTO TODO (TODO, TODO, TODO)
VALUES ('TODO', TODO, 'TODO');
"""

run_sql(sql)



# 7. 입력 결과 확인

데이터를 넣은 뒤에는 바로 `SELECT`로 확인하는 습관이 중요합니다.

## TODO 3

`students` 테이블의 모든 데이터를 조회하세요.


In [ ]:
sql = """
USE school;
SELECT TODO FROM TODO;
"""

run_sql(sql)



# 8. AUTO_INCREMENT 확인

방금 INSERT할 때 `id` 값을 직접 입력하지 않았습니다.

그런데 조회 결과를 보면 `id` 값이 자동으로 들어가 있습니다.

이것이:

```sql
AUTO_INCREMENT
```

의 역할입니다.

> 사람이 번호표를 직접 나눠 주는 것이 아니라,  
> MySQL이 자동으로 1, 2, 3, 4... 번호를 붙여 줍니다.

또한 `created_at`도 직접 입력하지 않았지만 현재 시간이 자동으로 저장됩니다.

그 이유는:

```sql
DEFAULT CURRENT_TIMESTAMP
```

가 설정되어 있기 때문입니다.



# 9. 여러 줄의 데이터를 한 번에 입력하기

한 번의 `INSERT INTO` 문으로 여러 행을 입력할 수 있습니다.

이번에 입력할 데이터:

| name | grade | class_name |
|---|---:|---|
| 혼이 | 1 | 영혼반 |
| 라즈베리 | 2 | 임베디드반 |
| 리눅스 | 3 | 서버반 |
| 파이썬 | 1 | 프로그래밍반 |

형식:

```sql
VALUES
(...),
(...),
(...),
(...);
```

## TODO 4

아래 SQL을 완성하세요.


In [ ]:
sql = """
USE school;

INSERT INTO students (name, grade, class_name)
VALUES
('TODO', TODO, 'TODO'),
('TODO', TODO, 'TODO'),
('TODO', TODO, 'TODO'),
('TODO', TODO, 'TODO');
"""

run_sql(sql)



# 10. 5개 레코드 확인

지금까지 정상적으로 입력했다면 다음 5명의 학생이 있어야 합니다.

- 김부장
- 혼이
- 라즈베리
- 리눅스
- 파이썬

## TODO 5

전체 데이터를 조회하세요.


In [ ]:
sql = """
USE school;
TODO * FROM TODO;
"""

run_sql(sql)



# 11. 컬럼 순서와 값의 순서

INSERT문에서는 **컬럼 순서와 값의 순서가 서로 맞아야 합니다.**

예:

```sql
INSERT INTO students (name, grade, class_name)
VALUES ('홍길동', 2, '데이터베이스반');
```

연결 관계:

```text
name       ← '홍길동'
grade      ← 2
class_name ← '데이터베이스반'
```

컬럼 순서를 바꾸는 것은 가능합니다.

```sql
INSERT INTO students (class_name, name, grade)
VALUES ('데이터베이스반', '홍길동', 2);
```

이것도 정상입니다.

하지만 다음처럼 값의 순서를 잘못 적으면 문제가 생깁니다.

```sql
INSERT INTO students (name, grade, class_name)
VALUES (2, '홍길동', '데이터베이스반');
```

> **항상 확인:**  
> 컬럼 순서와 값의 순서가 정확히 맞는가?


# 12. 컬럼 순서를 바꿔도 될까?

컬럼 순서를 바꾸는 것은 가능합니다. 중요한 것은 **컬럼 순서와 값의 순서가 서로 일치하는 것**입니다.

예를 들어 다음과 같이 작성할 수 있습니다.

```sql
INSERT INTO students (class_name, name, grade)
VALUES ('데이터베이스반', '홍길동', 2);
```

연결 관계는 다음과 같습니다.

```text
class_name ← '데이터베이스반'
name       ← '홍길동'
grade      ← 2
```

이번 단계는 **문법 이해 연습**입니다.  
뒤에서 `COUNT(*)` 결과가 원본 실습 자료와 같도록 이 SQL은 실제 테이블에 입력하지 않습니다.

## TODO 6

아래 SQL에서 값의 순서를 직접 완성한 뒤 출력 결과를 확인하세요.


In [ ]:
practice_sql = """
INSERT INTO students (class_name, name, grade)
VALUES ('TODO', 'TODO', TODO);
"""

if "TODO" in practice_sql:
    print("⚠️ TODO를 모두 완성하세요.")
else:
    print("작성한 SQL:")
    print(practice_sql)
    print("✅ 컬럼 순서와 값의 순서가 맞는지 스스로 확인하세요.")



# 13. 컬럼명을 생략할 수도 있을까?

다음처럼 컬럼명을 생략하는 문법도 가능합니다.

```sql
INSERT INTO students
VALUES (...);
```

하지만 초보자에게는 추천하지 않습니다.

왜냐하면 테이블의 **모든 컬럼 순서와 구조를 정확하게 기억해야 하기 때문**입니다.

`students`의 컬럼 순서는 다음과 같습니다.

```text
id
name
grade
class_name
created_at
```

예:

```sql
INSERT INTO students
VALUES (NULL, '정철이', 2, 'SQL반', NOW());
```

하지만 테이블 구조가 바뀌면 실수가 생길 수 있습니다.

> **초보 실습에서는 컬럼명을 적는 방식이 더 안전합니다.**



# 14. NOT NULL 오류 이해하기

`name` 컬럼은 다음처럼 만들어졌습니다.

```sql
name VARCHAR(50) NOT NULL
```

즉, 이름은 반드시 입력해야 합니다.

다음 SQL은 잘못된 예입니다.

```sql
INSERT INTO students (grade, class_name)
VALUES (2, '컴퓨터과');
```

이름이 빠졌기 때문에 오류가 발생할 수 있습니다.

---

## TODO 7 - 오류 예측

아래 문자열에 자신이 생각하는 오류 이유를 적으세요.


In [ ]:
reason = "TODO"

print("오류가 발생하는 이유:")
print(reason)



# 15. 일부 컬럼만 입력하기

모든 컬럼에 값을 넣을 필요는 없습니다.

다음처럼 이름만 입력할 수 있습니다.

```sql
INSERT INTO students (name)
VALUES ('최소입력');
```

이 경우:

- `id` → AUTO_INCREMENT
- `name` → '최소입력'
- `grade` → NULL
- `class_name` → NULL
- `created_at` → 현재 시간

이 됩니다.

### NULL이란?

`NULL`은 숫자 0이나 빈 문자열 `''`과 다릅니다.

```text
0    = 숫자 0
''   = 빈 문자열
NULL = 값이 없거나 아직 알 수 없음
```



## TODO 8 - 이름만 입력

이름이 `최소입력`인 레코드를 추가하세요.


In [ ]:
sql = """
USE school;

INSERT INTO students (TODO)
VALUES ('TODO');
"""

run_sql(sql)



# 16. 한글 데이터 입력 확인

이번 데이터베이스는 `utf8mb4` 문자 집합을 사용합니다.

따라서 한글도 정상적으로 저장할 수 있습니다.

입력할 데이터:

- 이름: 김코딩
- 학년: 1
- 반: 데이터베이스기초반

## TODO 9

아래 INSERT문을 완성하세요.


In [ ]:
sql = """
USE school;

INSERT INTO students (name, grade, class_name)
VALUES ('TODO', TODO, 'TODO');
"""

run_sql(sql)



# 17. 특정 학생만 조회하기

방금 입력한 `김코딩` 학생만 확인합니다.

형식:

```sql
SELECT *
FROM students
WHERE name = '이름';
```

## TODO 10

`김코딩` 학생만 조회하세요.


In [ ]:
sql = """
USE school;

SELECT * FROM students
WHERE TODO = 'TODO';
"""

run_sql(sql)



# 18. 입력한 데이터 개수 확인하기

테이블 안에 몇 개의 행이 있는지 확인할 때:

```sql
SELECT COUNT(*) FROM students;
```

를 사용할 수 있습니다.

`COUNT(*)`은 **전체 행 개수**를 셉니다.

## TODO 11

students 테이블의 전체 행 개수를 조회하세요.


In [ ]:
sql = """
USE school;

SELECT TODO FROM students;
"""

run_sql(sql)



# 19. 원하는 컬럼만 조회하기

모든 컬럼을 볼 필요가 없다면 필요한 컬럼만 선택할 수 있습니다.

예:

```sql
SELECT name, class_name
FROM students;
```

`*`는 모든 컬럼이라는 뜻입니다.

## TODO 12

학생 이름과 반/학과만 조회하세요.


In [ ]:
sql = """
USE school;

SELECT TODO, TODO
FROM students;
"""

run_sql(sql)


# 20. 종합 문제

다음 조건을 만족하는 INSERT문을 직접 작성하세요.

### 입력할 데이터

- 이름: 이몽룡
- 학년: 2
- 반: 컴퓨터과

### 조건

1. `school` 데이터베이스를 사용한다.
2. `students` 테이블에 입력한다.
3. 컬럼명은 반드시 명시한다.
4. `id`와 `created_at`은 직접 입력하지 않는다.
5. 입력 후 해당 학생만 조회한다.

## TODO 13

아래 코드 셀의 빈 공간에 SQL을 직접 작성하세요.  
`TODO`라는 글자를 SQL 안에 남겨 두지 않아도 됩니다.


In [ ]:
sql = """




"""

run_sql(sql)



# 21. TRUNCATE TABLE 이해하기

실습 중 데이터를 잘못 입력했거나 처음부터 다시 해보고 싶다면 테이블의 데이터만 모두 지울 수 있습니다.

```sql
TRUNCATE TABLE students;
```

### TRUNCATE TABLE

- 테이블 구조는 남아 있음
- 테이블 안 데이터는 모두 삭제
- `AUTO_INCREMENT` 번호도 다시 처음부터 시작

### DROP TABLE

```sql
DROP TABLE students;
```

- 테이블 자체를 삭제
- 구조도 사라짐
- 데이터도 사라짐

쉽게 비교하면:

```text
TRUNCATE TABLE → 서류철은 남기고 안의 서류만 모두 비움
DROP TABLE     → 서류철 자체를 버림
```

> ⚠️ `TRUNCATE TABLE`도 데이터가 모두 삭제되므로 실제 업무에서는 매우 주의해야 합니다.



# 22. 자주 발생하는 오류 정리

## ① 데이터베이스를 선택하지 않음

오류 예:

```text
No database selected
```

해결:

```sql
USE school;
```

---

## ② 테이블 이름 오타

우리 테이블 이름은:

```text
students
```

입니다.

`student`가 아닙니다.

확인:

```sql
SHOW TABLES;
```

---

## ③ 문자에 작은따옴표를 쓰지 않음

잘못된 예:

```sql
VALUES (김부장, 3, 컴퓨터과);
```

올바른 예:

```sql
VALUES ('김부장', 3, '컴퓨터과');
```

---

## ④ NOT NULL 컬럼을 생략

`name`은 반드시 입력해야 합니다.

잘못된 예:

```sql
INSERT INTO students (grade, class_name)
VALUES (3, '컴퓨터과');
```

올바른 예:

```sql
INSERT INTO students (name, grade, class_name)
VALUES ('홍길동', 3, '컴퓨터과');
```


# 23. 자동 점검

아래 셀은 현재 실습 결과를 자동으로 확인합니다.

모든 TODO를 **각각 한 번씩** 정상 실행했다면 최종적으로 다음 데이터가 존재합니다.

- 김부장
- 혼이
- 라즈베리
- 리눅스
- 파이썬
- 최소입력
- 김코딩
- 이몽룡(종합 문제까지 실행한 경우)

> INSERT 셀을 두 번 이상 실행하면 같은 이름의 레코드가 추가될 수 있습니다.  
> 이 경우 맨 아래의 초기화 셀을 실행한 뒤 처음부터 다시 진행하면 됩니다.


In [ ]:
import subprocess

def mysql_value(query):
    result = subprocess.run(
        ["sudo", "mysql", "-N", "-B", "-e", query],
        text=True,
        capture_output=True
    )
    if result.returncode != 0:
        return None
    return result.stdout.strip()

db_exists = mysql_value(
    "SELECT COUNT(*) FROM information_schema.SCHEMATA WHERE SCHEMA_NAME='school';"
)

table_exists = mysql_value(
    "SELECT COUNT(*) FROM information_schema.TABLES "
    "WHERE TABLE_SCHEMA='school' AND TABLE_NAME='students';"
)

print("===== 자동 점검 =====")
print("school 데이터베이스 :", "✅" if db_exists == "1" else "❌")
print("students 테이블     :", "✅" if table_exists == "1" else "❌")

if table_exists == "1":
    count = mysql_value("SELECT COUNT(*) FROM school.students;")
    base_names = ["김부장", "혼이", "라즈베리", "리눅스", "파이썬", "최소입력", "김코딩"]

    print("현재 전체 레코드 수 :", count)
    print("\n===== 필수 실습 데이터 =====")

    for name in base_names:
        n = mysql_value(
            "SELECT COUNT(*) FROM school.students "
            f"WHERE name='{name}';"
        )
        if n == "1":
            mark = "✅"
        elif n == "0":
            mark = "❌ 없음"
        else:
            mark = f"⚠️ {n}개(중복 실행 확인)"
        print(f"{name:<8} : {mark}")

    mong = mysql_value(
        "SELECT COUNT(*) FROM school.students WHERE name='이몽룡';"
    )
    print("\n종합 문제 이몽룡 :", 
          "✅" if mong == "1" else
          "아직 미실행" if mong == "0" else
          f"⚠️ {mong}개(중복)")



# 24. 오늘 배운 내용 정리

## INSERT INTO

```sql
INSERT INTO 테이블명 (컬럼1, 컬럼2, 컬럼3)
VALUES (값1, 값2, 값3);
```

핵심:

```text
INSERT INTO → 테이블에 데이터를 추가
문자 데이터 → 작은따옴표 사용
숫자 데이터 → 따옴표 없이 입력
컬럼 순서와 값의 순서 → 반드시 일치
여러 행 → 한 번의 INSERT문으로 입력 가능
AUTO_INCREMENT → id 자동 증가
NOT NULL → 반드시 값 입력
NULL → 값이 없거나 아직 알 수 없는 상태
SELECT → 입력 결과 확인
COUNT(*) → 전체 행 개수 확인
TRUNCATE TABLE → 구조는 유지하고 데이터만 모두 삭제
```

---

## 김부장식 정리

> **데이터베이스는 보관함,  
> 테이블은 서류철,  
> INSERT INTO는 서류철에 실제 기록을 끼워 넣는 작업입니다.**



# 25. 자기 점검

- [ ] `INSERT INTO`의 역할을 설명할 수 있다.
- [ ] 문자에는 작은따옴표를 사용해야 함을 알고 있다.
- [ ] 숫자는 따옴표 없이 입력할 수 있다.
- [ ] 한 줄 데이터를 입력할 수 있다.
- [ ] 여러 줄을 한 번에 입력할 수 있다.
- [ ] `AUTO_INCREMENT`의 역할을 설명할 수 있다.
- [ ] 컬럼 순서와 값의 순서가 맞아야 함을 알고 있다.
- [ ] 컬럼명을 생략하는 방식이 왜 위험할 수 있는지 설명할 수 있다.
- [ ] `NOT NULL` 오류가 발생하는 이유를 알고 있다.
- [ ] `NULL`, `0`, 빈 문자열의 차이를 설명할 수 있다.
- [ ] `COUNT(*)`으로 행 개수를 확인할 수 있다.
- [ ] 입력 후 `SELECT`로 결과를 확인할 수 있다.
- [ ] `TRUNCATE TABLE`과 `DROP TABLE`의 차이를 설명할 수 있다.


# 26. [선택] 데이터를 처음부터 다시 입력하기

아래 셀은 `students` 테이블의 **데이터만 모두 삭제**합니다.

- 테이블 구조는 유지됩니다.
- `AUTO_INCREMENT` 번호는 다시 1부터 시작합니다.
- 학생 데이터는 모두 삭제됩니다.

⚠️ 정말 초기화할 때만 `RESET = True`로 변경하여 실행하세요.

초기화 후에는 **6번 '데이터 한 줄 입력하기'부터 다시 진행**하면 됩니다.


In [ ]:
RESET = False

if RESET:
    !sudo mysql -e "USE school; TRUNCATE TABLE students;"
    print("🗑️ students 테이블의 데이터를 모두 삭제했습니다.")
    print("AUTO_INCREMENT 번호도 다시 처음부터 시작합니다.")
else:
    print("초기화하지 않았습니다.")
